In [ ]:
# =====================================================================
# CELL 1: INSTALASI, IMPORT & SETUP GROQ API
# =====================================================================
!pip install groq gradio pysastrawi numpy -q

from groq import Groq
import gradio as gr
import joblib
import re
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from google.colab import userdata

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

GROQ_API_KEY = userdata.get('GROQ_API_KEY')

if not GROQ_API_KEY:
    raise ValueError("⚠️ API Key tidak ditemukan!\nPastikan nama Secret adalah 'GROQ_API_KEY' dan Notebook access sudah ON.")

client_groq = Groq(api_key=GROQ_API_KEY)

print("✅ Setup Groq API berhasil!")
print("🚀 Model yang digunakan: Qwen 2.5 32B (via Groq LPU)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 6.1 MB/s eta 0:00:00
✅ Setup Groq API berhasil!
🚀 Model yang digunakan: Qwen 2.5 32B (via Groq LPU)


In [ ]:
from google.colab import files
import os
import joblib

print("📂 Upload file model_svm_best.pkl, tfidf_vectorizer.pkl, dan model_metadata.pkl:")
uploaded = files.upload()

# Validasi memastikan ketiga file .pkl berhasil diupload
required_files = ['model_svm_best.pkl', 'tfidf_vectorizer.pkl', 'model_metadata.pkl']
for f in required_files:
    if not os.path.exists(f):
        raise FileNotFoundError(f"⚠️ File {f} belum diupload! Silakan upload ketiga file .pkl tersebut.")

# Memuat model, vectorizer, dan metadata
model_svm = joblib.load('model_svm_best.pkl')
tfidf_vectorizer = joblib.load('tfidf_vectorizer.pkl')
meta = joblib.load('model_metadata.pkl')

print("✅ Model, Vectorizer, dan Metadata berhasil dimuat!")
print(f"   Kelas model     : {list(model_svm.classes_)}")
print(f"   Akurasi Test    : {meta['accuracy']*100:.2f}%")
print(f"   F1-Score (wtd)  : {meta['f1']*100:.2f}%")
print(f"   Jumlah Data     : {meta['jumlah_data']:,} ulasan")
print(f"   Konfigurasi     : {meta['split_scheme']} Split | {meta['smote']} ({meta['smote_sampling_strategy']})")

📂 Upload file model_svm_best.pkl, tfidf_vectorizer.pkl, dan model_metadata.pkl:


Saving model_metadata.pkl to model_metadata.pkl
Saving tfidf_vectorizer.pkl to tfidf_vectorizer.pkl
Saving model_svm_best.pkl to model_svm_best.pkl
✅ Model, Vectorizer, dan Metadata berhasil dimuat!
   Kelas model     : ['negatif', 'netral', 'positif']
   Akurasi Test    : 90.09%
   F1-Score (wtd)  : 88.76%
   Jumlah Data     : 2,673 ulasan
   Konfigurasi     : 80:20 Split | SVMSMOTE ({'netral': 300})


In [ ]:
# =====================================================================
# CELL 3: FUNGSI PREPROCESSING — WAJIB IDENTIK DENGAN NOTEBOOK TRAINING
# (kamus, stopword, dan urutan pipeline sama persis dengan
#  Skripsi_SVM_Fixed.ipynb agar token cocok dengan vocabulary TF-IDF)
# =====================================================================
KATA_NEGASI = {'tidak','tak','bukan','belum','tanpa','jangan','tiada','kurang'}

stopwords_id = set(stopwords.words('indonesian'))
stopwords_id.update({
    'yg','dgn','utk','krn','sdh','jg','sy','nih',
    'aja','deh','sih','dong','lah','kah','tuh',
    'google','play','store','android','ios',
    'bapak','ibu','pak','bu','mas','mbak','kak',
    'nya'
})
stopwords_id = stopwords_id - KATA_NEGASI

KAMUS_NORMALISASI = {
    'gak':'tidak','ga':'tidak','nggak':'tidak','ngga':'tidak',
    'gk':'tidak','tdk':'tidak','ndak':'tidak',
    'udah':'sudah','udh':'sudah','dah':'sudah','blm':'belum',
    'msh':'masih','masi':'masih',
    'emg':'memang','emang':'memang',
    'bgt':'banget','bngt':'banget',
    'bener':'benar','beneran':'benar',
    'tp':'tapi','tpi':'tapi',
    'klo':'kalau','klu':'kalau','kalo':'kalau',
    'lgi':'lagi','lg':'lagi',
    'krng':'kurang','krg':'kurang','bkn':'bukan',
    'gimana':'bagaimana','gmn':'bagaimana','knp':'kenapa',
    'oke':'baik','ok':'baik',
    'mantap':'bagus','mantul':'bagus','keren':'bagus',
    'susah':'sulit','payah':'buruk','parah':'buruk',
    'lelet':'lambat','lemot':'lambat','eror':'error',
    'loading':'memuat','update':'perbarui',
    'login':'masuk','logout':'keluar','hp':'ponsel',
}

stemmer = StemmerFactory().create_stemmer()

def cleaning(teks):
    teks = re.sub(r'http\S+|www\S+', '', teks)
    teks = re.sub(r'@\w+', '', teks)
    teks = re.sub(r'#\w+', '', teks)
    teks = teks.encode('ascii','ignore').decode('ascii')
    teks = re.sub(r'\d+', '', teks)
    teks = re.sub(r'[^\w\s]', ' ', teks)
    teks = re.sub(r'\s+', ' ', teks).strip()
    return teks

def case_folding(teks):
    return teks.lower()

def normalisasi(teks):
    return ' '.join([KAMUS_NORMALISASI.get(k, k) for k in teks.split()])

def tokenisasi(teks):
    return word_tokenize(teks)

def hapus_stopword(tokens):
    return [k for k in tokens if k not in stopwords_id and len(k) > 1]

def stemming(tokens):
    hasil = []
    for k in tokens:
        if k in KATA_NEGASI:
            hasil.append(k)  # kata negasi tidak di-stem
        else:
            hasil.append(stemmer.stem(k))
    return hasil

def preprocessing_lengkap(teks):
    if not isinstance(teks, str) or teks.strip() == '':
        return ''
    teks   = cleaning(teks)
    teks   = case_folding(teks)
    teks   = normalisasi(teks)
    tokens = tokenisasi(teks)
    tokens = hapus_stopword(tokens)
    tokens = stemming(tokens)
    return ' '.join(tokens)

# Sanity check: output WAJIB identik dengan demo di notebook training
print("✅ Preprocessing identik dengan notebook training!")
for c in ["apk nya tidak bagus, gak banget deh", "biasa aja sih, standar"]:
    print(f'  Input  : {c}\n  Output : {preprocessing_lengkap(c)}\n')

✅ Preprocessing identik dengan notebook training!
  Input  : apk nya tidak bagus, gak banget deh
  Output : apk tidak bagus tidak banget

  Input  : biasa aja sih, standar
  Output : standar



In [ ]:
# =====================================================================
# CELL 4: FUNGSI AI INSIGHT KOMPREHENSIF (DENGAN DETEKSI NETRAL PINTAR)
# =====================================================================
def generate_ai_insight(teks_asli, prediksi, confidence, rincian_proba=""):
    blok_proba = f"- Distribusi Probabilitas SVM: {rincian_proba}\n" if rincian_proba else ""

    prompt = f"""Anda adalah analis sentimen senior dan ahli linguistik komputasi.
Tugas Anda adalah menganalisis ulasan pengguna dan memberikan wawasan mendalam.

DATA INPUT:
- Teks Ulasan Asli: "{teks_asli}"
- Prediksi Model SVM: {prediksi.upper()} (Keyakinan: {confidence:.2f}%)
{blok_proba}

PENTING UNTUK DIMENGERTI:
Model SVM bekerja berdasarkan vektor kata dan bisa saja salah, terutama pada ulasan yang bersifat NETRAL (informatif/objektif) yang sering salah ditebak sebagai Negatif karena adanya kata "tidak". Gunakan pemahaman linguistik dan konteks bahasa Indonesia Anda untuk menilai sentimen ASLI dari teks tersebut, jangan hanya盲目 (blindly) mengikuti label SVM.

TUGAS ANDA (Berikan dalam format berikut):

📝 1. Analisis Konteks & Makna:
[Jelaskan makna tersirat. Apa yang sebenarnya ingin disampaikan pengguna?]

🎯 2. Aspek Layanan yang Dibahas:
[Identifikasi fitur: SIM, STNK, SKCK, ETLE, Pengaduan, Performa App, UI/UX, dll.]

😊 3. Analisis Emosi & Sentimen Asli:
[Analisis emosi dominan. JANGAN HANYA MENGIKUTI PREDIKSI SVM. Jika teks bersifat informatif, objektif, atau tidak memiliki emosi kuat, nyatakan dengan tegas bahwa ini adalah SENTIMEN NETRAL, meskipun SVM memprediksi Positif/Negatif.]

🔍 4. Identifikasi Masalah (Jika Ada):
[Jika ada keluhan, identifikasi: Bug, Lambat, UI Bingung, dll. Jika tidak ada, tulis "Tidak ada masalah".]

⚠️ 5. Tingkat Prioritas & Justifikasi:
[Berikan angka 1-5:
1 = Rendah (pujian/normal)
2 = Sedang (masukan minor)
3 = Cukup Penting (perlu diperhatikan)
4 = Penting (masalah fitur inti)
5 = Mendesak (bug kritis)]

 6. Dampak terhadap Pengguna:
[Seberapa besar dampak masalah/fitur ini terhadap pengalaman pengguna?]

🧭 7. Evaluasi Kecocokan dengan Model SVM:
[Bandingkan sentimen asli Anda dengan prediksi SVM ({prediksi.upper()}). Jelaskan apakah SVM sudah benar atau salah menebak. Jika SVM salah (misal: teks netral ditebak negatif), jelaskan mengapa SVM bisa terjebak (misal: karena kata "tidak").]

ATURAN PENTING:
- Gunakan bahasa Indonesia profesional, analitis, dan mudah dipahami.
- Objektif dan berbasis data dari teks.
- Format markdown rapi; total 300-450 kata."""

    try:
        completion = client_groq.chat.completions.create(
            # Ganti dengan model yang Anda pakai saat ini (misal: openai/gpt-oss-120b atau qwen-2.5-32b)
            model="openai/gpt-oss-120b",
            messages=[
                {
                    "role": "system",
                    "content": """Anda adalah ahli linguistik dan analis sentimen ahli. Anda mampu memahami nuansa bahasa Indonesia, termasuk sarkasme, kalimat informatif, dan netralitas. Berikan analisis yang jujur berdasarkan teks asli, bukan hanya berdasarkan label machine learning."""
                },
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            max_tokens=1000,
            top_p=0.9
        )
        return completion.choices[0].message.content
    except Exception as e:
        return f"""⚠️ Gagal menghasilkan insight AI. Error: {str(e)}"""

print("✅ Fungsi AI Insight siap! (Dioptimasi untuk deteksi Netral & Koreksi SVM)")

✅ Fungsi AI Insight siap! (Dioptimasi untuk deteksi Netral & Koreksi SVM)


In [ ]:
# =====================================================================
# CELL 5: FUNGSI PREDIKSI + AI INSIGHT (DIPERBAIKI)
# =====================================================================
import numpy as np

def prediksi_dengan_insight(teks_ulasan):
    """Fungsi utama: prediksi SVM + AI Insight Groq."""
    if not teks_ulasan or len(teks_ulasan.strip()) == 0:
        return "⚠️ Silakan masukkan teks ulasan terlebih dahulu!"

    # 1. Preprocessing
    try:
        teks_bersih = preprocessing_lengkap(teks_ulasan)
    except NameError:
        teks_bersih = preprocessing(teks_ulasan)

    # 2. TF-IDF Transform
    vektor = tfidf_vectorizer.transform([teks_bersih])

    # 3. Prediksi SVM
    pred = model_svm.predict(vektor)[0]

    # 4. Confidence
    proba = model_svm.predict_proba(vektor)[0]
    kelas_urut = list(model_svm.classes_)
    confidence = proba[kelas_urut.index(pred)] * 100

    # 5. Distribusi probabilitas
    rincian_proba = ", ".join(
        [f"{k}: {proba[kelas_urut.index(k)]*100:.2f}%" for k in ['positif', 'netral', 'negatif']]
    )

    # 6. Catatan arah
    urut_proba = sorted(proba, reverse=True)
    margin = urut_proba[0] - urut_proba[1]
    p_netral = proba[kelas_urut.index('netral')]
    if pred != 'netral' and (margin < 0.15 or p_netral > 0.30):
        catatan_arah = "⚠️ Margin probabilitas tipis — ulasan ini cenderung **ambigu/netral**. Prediksi di atas adalah kecenderungan terdekat."
    else:
        catatan_arah = "✅ Model relatif yakin pada prediksi ini."

    # 7. Emoji mapping
    emoji_map = {'positif': '😊 POSITIF', 'netral': '😐 NETRAL', 'negatif': '😠 NEGATIF'}

    # 8. AI Insight via Groq
    try:
        insight = generate_ai_insight(teks_ulasan, pred, confidence, rincian_proba)
    except:
        insight = "️ Gagal menghasilkan AI Insight. Periksa koneksi API Groq."

    # 9. Format output - STRUKTUR YANG BENAR SESUAI WIREFRAME
    hasil = f"""
## 📊 Hasil Analisis Sentimen Super App Polri

###  Klasifikasi Model (SVM + SMOTE + GridSearchCV)

| Informasi | Detail |
|-----------|--------|
| **Teks Asli** | "{teks_ulasan}" |
| **Teks Setelah Preprocessing** | "{teks_bersih}" |
| **Prediksi Sentimen** | {emoji_map[pred]} |
| **Probabilitas Prediksi Model** | {confidence:.2f}% |
| **Distribusi Probabilitas** | {rincian_proba} |

**🧭 Catatan Arah Sentimen:** {catatan_arah}

---

### 🧠 AI Insight (Powered by Groq - GPT-OSS 120B)

{insight}

---
*Analisis dilakukan oleh model SVM + Groq AI untuk interpretasi kontekstual.*
"""
    return hasil

print("✅ Fungsi prediksi + insight siap!")

✅ Fungsi prediksi + insight siap!


In [ ]:
# =====================================================================
# CELL 6: JALANKAN GRADIO DENGAN AI INSIGHT
# =====================================================================
iface = gr.Interface(
    fn=prediksi_dengan_insight,
    inputs=gr.Textbox(
        lines=4,
        placeholder="Masukkan ulasan pengguna Super App Presisi Polri di sini...\n\nContoh: 'Aplikasi bagus tapi sering error saat perpanjang SIM'",
        label="📝 Teks Ulasan (Input Manual)"
    ),
    outputs=gr.Markdown(
        label="📊 Hasil Analisis + AI Insight"
    ),
    title="🚔 Sistem Analisis Sentimen Super App Presisi Polri",
    description="""
**Arsitektur Sistem:**
- **Model Klasifikasi:** Support Vector Machine (SVM) + Grid Search
- **AI Interpretasi:** GPT-OSS 120B (via API)
- **Kelas Sentimen:** Positif | Netral | Negatif

Sistem ini menggabungkan kekuatan SVM untuk klasifikasi cepat dengan GPT-OSS 120B untuk memberikan insight kontekstual dan rekomendasi bagi pengembang aplikasi.
""",
    article="""
---
**Dikembangkan oleh:** Carel Alberto Karma
**Metode:** SVM + Grid Search + GPT-OSS 120B
**Dataset:** Ulasan Google Play Store (Juni–Agustus 2026)
**Package Name:** superapps.polri.presisi.presisi
**Preprocessing:** Case Folding + Cleaning + Normalisasi + Tokenisasi + Stopword Removal + Stemming
""",
    examples=[
        ["Aplikasinya sangat bagus, perpanjangan SIM jadi lebih mudah dan cepat!"],
        ["Sering error dan lemot, tidak bisa dibuka sama sekali saat mau bayar STNK."],
        ["Bagaimana cara perpanjang STNK di aplikasi ini? Masih bingung menu nya."],
        ["Fitur pengaduan masyarakat sangat membantu, tapi loadingnya agak lama."],
        ["Aplikasi crash terus, sudah coba install ulang tetap sama. Kecewa banget!"],
        ["Lumayan bagus tapi kadang ngelag saat upload foto SKCK."]
    ]
)

print("🚀 Meluncurkan Gradio dengan AI Insight...")
iface.launch(share=True, debug=True)

🚀 Meluncurkan Gradio dengan AI Insight...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://65c795a46c00981891.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
